# Restaurant Rating Prediction using Machine Learning
**Domain: Data Science & Machine Learning**

### Project Overview
This project builds an end-to-end Machine Learning pipeline to predict the **Aggregate Rating** of restaurants using historical data from the Zomato platform. We compare several machine learning algorithms, including traditional regressions, tree ensembles, XGBoost, and Deep Learning (TensorFlow/Keras and PyTorch).

### Author
Senior Machine Learning Engineer & Data Scientist

## Setup & Library Imports
We first import all required standard and specialized libraries for numerical operations, data manipulation, visualization, modeling, and deep learning.

In [ ]:
%matplotlib inline
import os
import sys
import pickle
import joblib
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

from sklearn.model_selection import train_test_split, KFold, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor

try:
    import xgboost as xgb
    XGB_AVAILABLE = True
except ImportError:
    XGB_AVAILABLE = False
    print("XGBoost not available.")

try:
    import tensorflow as tf
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
    from tensorflow.keras.callbacks import EarlyStopping
    TF_AVAILABLE = True
except ImportError:
    TF_AVAILABLE = False
    print("TensorFlow not available.")

try:
    import torch
    import torch.nn as nn
    import torch.optim as optim
    from torch.utils.data import TensorDataset, DataLoader
    TORCH_AVAILABLE = True
except ImportError:
    TORCH_AVAILABLE = False
    print("PyTorch not available.")

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

## Part 2: Loading & Profiling the Dataset
We load the dataset using Pandas. Note that the Zomato dataset contains special local characters, so we read it with `latin-1` encoding to prevent encoding failures.

In [ ]:
df = pd.read_csv('dataset/restaurant_data.csv', encoding='latin-1')
df.columns = df.columns.str.replace('ï»¿', '').str.replace('\\ufeff', '').str.strip()
print(f"Shape: {df.shape}")
df.info()

In [ ]:
print("Missing Values:")
print(df.isnull().sum()[df.isnull().sum() > 0])
print(f"Duplicates: {df.duplicated().sum()}")

## Part 2 & 4: Data Preprocessing & Feature Engineering
We apply standard cleanups:
1. Remove duplicates.
2. Handle missing Cuisines values.
3. Engineer features: `Cuisine Count`, `Online Delivery Flag`, `Table Booking Flag`, `Restaurant Age` (simulated), `Cost Category`, log transformations of `Votes` and `Average Cost for two`.
4. Drop Target Leakage and Redundant Columns. Columns like `Rating color` and `Rating text` have direct mapping from `Aggregate rating` and represent a severe target leakage. We drop them to keep the model realistic.
5. Label encode categoricals.

In [ ]:
# Clean duplicates and handle missing values
df_clean = df.copy()
df_clean.drop_duplicates(inplace=True)
df_clean['Cuisines'] = df_clean['Cuisines'].fillna('Unknown Cuisines')

# Feature Engineering
df_clean['Cuisine Count'] = df_clean['Cuisines'].apply(lambda x: len(str(x).split(',')))
df_clean['Online Delivery Flag'] = df_clean['Has Online delivery'].map({'Yes': 1, 'No': 0})
df_clean['Table Booking Flag'] = df_clean['Has Table booking'].map({'Yes': 1, 'No': 0})
df_clean['Restaurant Age'] = 2026 - (df_clean['Restaurant ID'] % 15 + 2010)

def categorize_cost(cost):
    if cost <= 300: return 'Low'
    elif cost <= 800: return 'Medium'
    elif cost <= 2000: return 'High'
    else: return 'Premium'
df_clean['Cost Category'] = df_clean['Average Cost for two'].apply(categorize_cost)
df_clean['Price Bucket'] = df_clean['Price range'].astype(float)
df_clean['Log Votes'] = np.log1p(df_clean['Votes'])
df_clean['Log Cost'] = np.log1p(df_clean['Average Cost for two'])

# Target Leakage Mitigation & Redundancy Drop
columns_to_drop = [
    'Restaurant ID', 'Restaurant Name', 'Address', 'Locality Verbose', 
    'Currency', 'Switch to order menu', 'Rating color', 'Rating text'
]
df_clean.drop(columns=columns_to_drop, inplace=True, errors='ignore')

# Encoding Categorical Variables
categorical_cols = ['City', 'Locality', 'Cuisines', 'Cost Category', 'Has Online delivery', 'Has Table booking', 'Is delivering now']
label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    df_clean[col] = le.fit_transform(df_clean[col].astype(str))
    label_encoders[col] = le

df_clean.head()

## Part 3: Exploratory Data Analysis (EDA)
We visualize distributions, boxplots, correlation heatmaps, and pairplots.

In [ ]:
# 1. Rating Distribution
plt.figure(figsize=(10, 5))
sns.histplot(df_clean['Aggregate rating'], kde=True, bins=30, color='darkblue')
plt.title('Distribution of Restaurant Aggregate Ratings')
plt.xlabel('Aggregate Rating')
plt.ylabel('Count')
plt.show()

In [ ]:
# 2. Votes vs Rating
plt.figure(figsize=(10, 5))
sns.scatterplot(x='Votes', y='Aggregate rating', data=df, alpha=0.5, color='teal')
plt.title('Votes vs Aggregate Rating')
plt.xlabel('Votes')
plt.ylabel('Aggregate Rating')
plt.show()

In [ ]:
# 3. Cost vs Rating
plt.figure(figsize=(10, 5))
sns.boxplot(x='Price range', y='Aggregate rating', data=df, palette='viridis')
plt.title('Aggregate Rating by Price Range')
plt.xlabel('Price Range')
plt.ylabel('Aggregate Rating')
plt.show()

In [ ]:
# 4. Correlation Heatmap
plt.figure(figsize=(12, 10))
sns.heatmap(df_clean.corr(), annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Correlation Matrix of Processed Features')
plt.show()

## Train-Test Split and Standardization

In [ ]:
X = df_clean.drop(columns=['Aggregate rating'])
y = df_clean['Aggregate rating']
feature_names = X.columns.tolist()

X_train_raw, X_test_raw, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)
X_test = scaler.transform(X_test_raw)
scaler.feature_names_in_ = np.array(feature_names)

## Part 5 & 6: Train Traditional ML Models

In [ ]:
models = {
    'Linear Regression': LinearRegression(),
    'Decision Tree': DecisionTreeRegressor(max_depth=8, random_state=42),
    'Random Forest': RandomForestRegressor(n_estimators=100, max_depth=12, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=5, random_state=42),
    'Extra Trees': ExtraTreesRegressor(n_estimators=100, max_depth=12, random_state=42, n_jobs=-1)
}

if XGB_AVAILABLE:
    models['XGBoost'] = xgb.XGBRegressor(n_estimators=100, max_depth=6, learning_rate=0.1, random_state=42, n_jobs=-1)

trained_models = {}
evaluation_results = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    
    mae = mean_absolute_error(y_test, preds)
    mse = mean_squared_error(y_test, preds)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, preds)
    n, p = len(y_test), X_train.shape[1]
    adj_r2 = 1 - (1 - r2) * (n - 1) / (n - p - 1)
    
    cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='r2')
    cv_mean = cv_scores.mean()
    
    evaluation_results[name] = {
        'MAE': mae, 'MSE': mse, 'RMSE': rmse,
        'R2 Score': r2, 'Adjusted R2': adj_r2, 'CV Score (R2)': cv_mean
    }
    trained_models[name] = model
    print(f"{name:<20} | R2: {r2:.4f} | RMSE: {rmse:.4f}")

## Part 5 & 6: Train Deep Learning Models (TensorFlow & Keras)

In [ ]:
if TF_AVAILABLE:
    target_scaler = StandardScaler()
    y_train_scaled = target_scaler.fit_transform(y_train.values.reshape(-1, 1)).flatten()
    
    # 7. TF DNN Regression Model
    tf_model = Sequential([
        Dense(128, activation='relu', input_shape=(X_train.shape[1],)),
        BatchNormalization(),
        Dropout(0.2),
        Dense(64, activation='relu'),
        BatchNormalization(),
        Dropout(0.2),
        Dense(32, activation='relu'),
        Dense(1)
    ])
    tf_model.compile(optimizer='adam', loss='mse', metrics=['mae'])
    early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
    tf_model.fit(X_train, y_train_scaled, epochs=100, batch_size=64, validation_split=0.1, callbacks=[early_stop], verbose=0)
    
    preds_scaled = tf_model.predict(X_test, verbose=0).flatten()
    predictions = target_scaler.inverse_transform(preds_scaled.reshape(-1, 1)).flatten()
    
    r2 = r2_score(y_test, predictions)
    rmse = np.sqrt(mean_squared_error(y_test, predictions))
    evaluation_results['TensorFlow DNN'] = {
        'MAE': mean_absolute_error(y_test, predictions),
        'MSE': mean_squared_error(y_test, predictions),
        'RMSE': rmse,
        'R2 Score': r2,
        'Adjusted R2': 1 - (1 - r2) * (len(y_test) - 1) / (len(y_test) - X_train.shape[1] - 1),
        'CV Score (R2)': np.nan
    }
    
    # 8. Keras Sequential Model (Alternative)
    keras_alt = Sequential([
        Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
        Dropout(0.1),
        Dense(32, activation='relu'),
        Dense(1)
    ])
    keras_alt.compile(optimizer=tf.keras.optimizers.RMSprop(learning_rate=0.001), loss='mse')
    keras_alt.fit(X_train, y_train_scaled, epochs=50, batch_size=64, validation_split=0.1, verbose=0)
    alt_preds_scaled = keras_alt.predict(X_test, verbose=0).flatten()
    alt_predictions = target_scaler.inverse_transform(alt_preds_scaled.reshape(-1, 1)).flatten()
    alt_r2 = r2_score(y_test, alt_predictions)
    evaluation_results['Keras Sequential'] = {
        'MAE': mean_absolute_error(y_test, alt_predictions),
        'MSE': mean_squared_error(y_test, alt_predictions),
        'RMSE': np.sqrt(mean_squared_error(y_test, alt_predictions)),
        'R2 Score': alt_r2,
        'Adjusted R2': 1 - (1 - alt_r2) * (len(y_test) - 1) / (len(y_test) - X_train.shape[1] - 1),
        'CV Score (R2)': np.nan
    }
    print(f"TensorFlow DNN     | R2: {r2:.4f} | RMSE: {rmse:.4f}")
    print(f"Keras Alt          | R2: {alt_r2:.4f}")

## Part 5 & 6: Train PyTorch Regression Model

In [ ]:
if TORCH_AVAILABLE:
    y_scaler = StandardScaler()
    y_train_scaled = y_scaler.fit_transform(y_train.values.reshape(-1, 1))
    
    X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
    y_train_tensor = torch.tensor(y_train_scaled, dtype=torch.float32)
    X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
    
    dataset = TensorDataset(X_train_tensor, y_train_tensor)
    loader = DataLoader(dataset, batch_size=64, shuffle=True)
    
    class PyTorchRegressor(nn.Module):
        def __init__(self, input_dim):
            super(PyTorchRegressor, self).__init__()
            self.net = nn.Sequential(
                nn.Linear(input_dim, 64),
                nn.ReLU(),
                nn.Linear(64, 32),
                nn.ReLU(),
                nn.Linear(32, 1)
            )
        def forward(self, x): return self.net(x)
        
    pt_model = PyTorchRegressor(X_train.shape[1])
    criterion = nn.MSELoss()
    optimizer = optim.Adam(pt_model.parameters(), lr=0.005)
    
    pt_model.train()
    for epoch in range(100):
        for bx, by in loader:
            optimizer.zero_grad()
            outputs = pt_model(bx)
            loss = criterion(outputs, by)
            loss.backward()
            optimizer.step()
            
    pt_model.eval()
    with torch.no_grad():
        preds_scaled = pt_model(X_test_tensor).numpy()
        predictions = y_scaler.inverse_transform(preds_scaled).flatten()
        
    r2 = r2_score(y_test, predictions)
    rmse = np.sqrt(mean_squared_error(y_test, predictions))
    evaluation_results['PyTorch MLP'] = {
        'MAE': mean_absolute_error(y_test, predictions),
        'MSE': mean_squared_error(y_test, predictions),
        'RMSE': rmse,
        'R2 Score': r2,
        'Adjusted R2': 1 - (1 - r2) * (len(y_test) - 1) / (len(y_test) - X_train.shape[1] - 1),
        'CV Score (R2)': np.nan
    }
    print(f"PyTorch MLP        | R2: {r2:.4f} | RMSE: {rmse:.4f}")

## Model Comparison Table

In [ ]:
pd.DataFrame(evaluation_results).T

## Part 8 & 9: Hyperparameter Tuning and Model Saving
We identify the best ML model based on R² Score and tune it using GridSearchCV. We then serialize the model using pickle.

In [ ]:
valid_ml = {k: v for k, v in evaluation_results.items() if not np.isnan(v['CV Score (R2)'])} 
best_name = max(valid_ml, key=lambda k: valid_ml[k]['R2 Score'])
print(f"Tuning: {best_name}")

if best_name in ['Random Forest', 'Extra Trees']:
    base = RandomForestRegressor(random_state=42, n_jobs=-1) if best_name == 'Random Forest' else ExtraTreesRegressor(random_state=42, n_jobs=-1)
    grid = GridSearchCV(base, {'n_estimators': [100, 150], 'max_depth': [10, 15]}, cv=3, scoring='r2', n_jobs=-1)
    grid.fit(X_train, y_train)
    best_tuned = grid.best_estimator_
    print(f"Best Parameters: {grid.best_params_}")
else:
    best_tuned = trained_models[best_name]
    print("Using default base model.")

## Save Deployment Artifacts

In [ ]:
with open('models/best_model.pkl', 'wb') as f:
    pickle.dump({'model': best_tuned, 'scaler': scaler, 'encoders': label_encoders}, f)
print("Best model saved successfully!")

## Part 10: Inference on New Sample Data

In [ ]:
with open('models/best_model.pkl', 'rb') as f:
    artifacts = pickle.load(f)
    
sample_input = {
    'City': 'New Delhi', 'Locality': 'Connaught Place', 'Cuisines': 'North Indian, Chinese',
    'Average Cost for two': 1200, 'Has Table booking': 'Yes', 'Has Online delivery': 'Yes',
    'Is delivering now': 'No', 'Price range': 3, 'Votes': 450
}

sample_proc = {
    'Cuisine Count': len(sample_input['Cuisines'].split(',')),
    'Online Delivery Flag': 1 if sample_input['Has Online delivery'] == 'Yes' else 0,
    'Table Booking Flag': 1 if sample_input['Has Table booking'] == 'Yes' else 0,
    'Restaurant Age': 5,
    'Cost Category': 'High',
    'Price Bucket': float(sample_input['Price range']),
    'Log Votes': np.log1p(sample_input['Votes']),
    'Log Cost': np.log1p(sample_input['Average Cost for two']),
    'City': sample_input['City'], 'Locality': sample_input['Locality'], 'Cuisines': sample_input['Cuisines'],
    'Has Online delivery': sample_input['Has Online delivery'], 'Has Table booking': sample_input['Has Table booking'],
    'Is delivering now': sample_input['Is delivering now'], 'Price range': sample_input['Price range'],
    'Average Cost for two': sample_input['Average Cost for two'], 'Votes': sample_input['Votes']
}

sample_df = pd.DataFrame([sample_proc])
for col, enc in artifacts['encoders'].items():
    sample_df[col] = enc.transform([str(sample_df.loc[0, col])])[0]
    
sample_scaled = artifacts['scaler'].transform(sample_df[artifacts['scaler'].feature_names_in_])
predicted_r = artifacts['model'].predict(sample_scaled)[0]
print(f"Predicted Aggregate Rating: {predicted_r:.2f} / 5.0")